# Day I7 — 보행의 신경 조절과 이중과제 비용 + 1주차 정리

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I7.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 신경 조절 지도의 뼈대 그리기

아래 코드는 지도의 **뼈대**만 그립니다. 오늘 직접 해보기에서 화살표와 설명을 더해 완성합니다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# 상자 이름과 (x, y) 위치. 그림 글자는 영어(한글 글꼴 문제 회피)
nodes = {
    "Cortex\n(M1, SMA/PM,\ntemporoparietal)": (0.5, 0.92),
    "Basal ganglia\n(SNr/GPi output)": (0.15, 0.68),
    "Cerebellum\n(error correction)": (0.85, 0.68),
    "Brainstem\nMLR (CNF, PPN)": (0.5, 0.48),
    "Reticulospinal\ntract": (0.5, 0.28),
    "Spinal locomotor\nnetwork (CPG)": (0.5, 0.08),
    "Sensory feedback\n(proprioception,\nvision, vestibular)": (0.88, 0.2),
}
edges = [  # (출발, 도착, 선 모양) : "-" 흥분/조절, "--" 억제(GABA)
    ("Cortex\n(M1, SMA/PM,\ntemporoparietal)", "Brainstem\nMLR (CNF, PPN)", "-"),
    ("Basal ganglia\n(SNr/GPi output)", "Brainstem\nMLR (CNF, PPN)", "--"),
    ("Cerebellum\n(error correction)", "Brainstem\nMLR (CNF, PPN)", "-"),
    ("Brainstem\nMLR (CNF, PPN)", "Reticulospinal\ntract", "-"),
    ("Reticulospinal\ntract", "Spinal locomotor\nnetwork (CPG)", "-"),
    ("Sensory feedback\n(proprioception,\nvision, vestibular)", "Spinal locomotor\nnetwork (CPG)", "-"),
    ("Sensory feedback\n(proprioception,\nvision, vestibular)", "Cerebellum\n(error correction)", "-"),
]
fig, ax = plt.subplots(figsize=(7, 6))
for name, (x, y) in nodes.items():
    ax.text(x, y, name, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round", fc="white", ec="k"))
for a, b, style in edges:
    (x0, y0), (x1, y1) = nodes[a], nodes[b]
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="->", ls=style, shrinkA=28, shrinkB=28))
ax.text(0.02, 0.02, "automatic (brainstem-spinal) vs executive (cortical) control\n"
        "dashed = inhibitory output", transform=ax.transAxes, va="bottom", fontsize=8)
ax.axis("off")
out = Path("reports/advanced/I/I7_neural_map.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=150)
print("저장:", out)

### 예시 2 — gaitpdb로 이중과제 비용 계산하기

> **데이터:** Hausdorff JM. *Gait in Parkinson's Disease* (v1.0.0). PhysioNet; 2008. [doi:10.13026/C24H3N](https://doi.org/10.13026/C24H3N). Open Data Commons Attribution License v1.0. PhysioNet 표준 인용: Pollard T, et al. *Nature Health* (2026). [doi:10.1038/s44360-026-00096-z](https://doi.org/10.1038/s44360-026-00096-z). `demographics.txt`의 `Group`은 1 = PD, 2 = 대조군이고(`.html`과 대조해 확인), `Speed_01`은 평소 걷기, `Speed_10`은 Ga 연구의 이중과제(걸으며 7씩 빼기) 걷기 속도(m/s)입니다. 내려받기는 [Day I4](day-I4.md)와 같습니다(`demographics.txt`만 있으면 됩니다).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 줄 끝 빈칸 때문에 앞 20열만 읽는다(Day 25에서 본 ParserError 회피)
demo = pd.read_csv("data/gaitpdb/demographics.txt", sep="\t", usecols=range(20))
demo["group"] = demo["Group"].map({1: "PD", 2: "Control"})      # .html과 대조해 확인한 코드

print("Speed_10 값이 있는 연구:", sorted(demo.loc[demo["Speed_10"].notna(), "Study"].unique()))
ga = demo[(demo["Study"] == "Ga") & demo["Speed_01"].notna() & demo["Speed_10"].notna()].copy()
print("두 속도가 모두 있는 Ga 참가자:", ga["group"].value_counts().to_dict())

# 흔히 쓰는 정의: 이중과제 비용(%) = (평소 − 이중과제) / 평소 × 100  (클수록 이중과제에서 더 느려짐)
ga["DTC"] = (ga["Speed_01"] - ga["Speed_10"]) / ga["Speed_01"] * 100

summary = ga.groupby("group").agg(n=("DTC", "size"),
                                  usual=("Speed_01", "median"),
                                  dual=("Speed_10", "median"),
                                  DTC_median=("DTC", "median"),
                                  DTC_q1=("DTC", lambda s: s.quantile(0.25)),
                                  DTC_q3=("DTC", lambda s: s.quantile(0.75)),
                                  slower_n=("DTC", lambda s: int((s > 0).sum())))
print(summary.round(2).to_string())

fig, ax = plt.subplots(figsize=(4.5, 3.6))
rng = np.random.default_rng(0)                                   # 점이 겹치지 않게 가로로 살짝 흩뿌림
for k, (g, sub) in enumerate(ga.groupby("group")):
    ax.scatter(k + rng.uniform(-0.08, 0.08, len(sub)), sub["DTC"], alpha=0.7)
    ax.hlines(sub["DTC"].median(), k - 0.25, k + 0.25, color="k")
ax.axhline(0, color="gray", lw=0.6)
ax.set_xticks([0, 1], [f"{g} (n={n})" for g, n in ga["group"].value_counts().sort_index().items()])
ax.set_ylabel("dual-task cost of gait speed (%)")
ax.set_title("gaitpdb v1.0.0, Ga study only")
out = Path("figures/advanced/I/I7_dual_task_cost.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=150)
print("저장:", out)

## 🤖 AI와 함께 하기 (주간 실습, 20–30분)

3. **AI 코드를 "정답을 아는 가상 자세"로 시험합니다.** 아래는 이번 주 규약(굴곡 +)으로 쓴 함수를 시험하는 예입니다. AI가 준 함수로 `knee_angle`을 바꿔 넣고 같은 세 자세를 넣어 보세요.

In [ ]:
import numpy as np

def knee_angle(hip, knee, ankle):
    """AI가 써 준 함수라고 가정하고, 정답을 아는 자세로 시험한다 (x = 앞, y = 위)."""
    thigh = np.degrees(np.arctan2(hip[1] - knee[1], hip[0] - knee[0]))
    leg = np.degrees(np.arctan2(knee[1] - ankle[1], knee[0] - ankle[0]))
    return thigh - leg

# 정답을 아는 가상 자세 1: 다리를 곧게 편 채 수직으로 섬 → 0°
print("곧게 선 자세  :", round(knee_angle((0, 0.9), (0, 0.5), (0, 0.1)), 1))
# 가상 자세 2: 대퇴는 수직, 하퇴가 뒤로 30° 기움(무릎이 굽음) → +30°가 나와야 굴곡 + 규약
d = 0.4
ankle = (-d * np.sin(np.radians(30)), 0.5 - d * np.cos(np.radians(30)))
print("무릎 30° 굽힘 :", round(knee_angle((0, 0.9), (0, 0.5), ankle), 1))
# 가상 자세 3: 반대쪽(왼쪽)을 보고 걷는 영상이라 x가 뒤집힌 경우 → 부호가 뒤집힌다
print("좌우 뒤집힌 영상:", round(knee_angle((0, 0.9), (0, 0.5), (-ankle[0], ankle[1])), 1))